In [11]:
import pandas as pd
import numpy as np
import re

# =========================
# 1. CARGA DEL CSV
# =========================

df = pd.read_csv(
    "limpieza_demanda_real_completa_2020_2024.csv",
    sep=";",
    parse_dates=["datetime"]
)

# =========================
# 2. LIMPIEZA DE VALUE
# =========================

#funcion para eliminar los puntos execepto el primero, que indica los decimales
def clean_mixed_number(x):
    if x is None:
        return x
    
    s = str(x).strip()
    
    # Si no tiene puntos o solo hay uno, no hace falta hacer nada
    if s.count(".") <= 1:
        return float(s)
    
    # Dividimos la cadena por los puntos
    parts = s.split(".")
    
    # Unimos todo menos la ultima parte (los deciamles son a partir del primer punto a la derecha)
    integer_part = "".join(parts[:-1])
    decimal_part = parts[-1]
    
    cleaned = integer_part + "." + decimal_part
    
    # Si no tiene punto
    return float(cleaned)


# Aplicar la funcion a la columna value
df["value"] = df["value"].apply(clean_mixed_number)

# Eliminar filas con valores invalidos (na)
df = df.dropna(subset=["value"])

# =========================
# 3. LIMPIEZA DATETIME
# ==========================

df["datetime"] = df["datetime"].str.replace("T", " ", regex=False)
df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

# =========================
# 3. Eliminar Columna id
# ==========================

df = df.drop('id', axis=1)

# Convertir la columna datetime a datetime y establecerla como índice
#df = df.set_index('datetime')

df.to_csv("LimpizaGeneracionMediaTipos2020_2024V1.csv", index=False, sep=';')
